# DBiT-seq dataset 30258697

**Publication:** [Spatial CUT&Tag — Nature Methods 2024](https://www.nature.com/articles/s41592-024-02576-0)

Mouse E13 embryo, multi-modal spatial (RNA + histone marks + ATAC).

Three sample groups:
- **E13_20_fig3** (20 µm): RNA, H3K27me3, H3K4me3
- **E13_50_2** (50 µm): RNA, ATAC, H3K27me3, H3K4me3
- **E13_50_1** (50 µm): H3K27ac, H3K27me3 (no RNA)

**Strategy:** Concatenate all modalities along the var axis with prefixed gene names
(e.g. `Sox17`, `H3K27me3:Sox17`, `ATAC:Sox17`) so every feature is in X and
visible in KaroSpace.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from scipy import sparse
import matplotlib.pyplot as plt

DATA_DIR = Path('/Volumes/processing2/30258697')
OUT_DIR = Path('/Volumes/processing2/KaroSpaceDataWrangle/dbit-30258697')
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_DIR.exists(), f'Data dir not found: {DATA_DIR}'

## 1. File discovery & grouping

In [ ]:
h5ad_files = sorted(DATA_DIR.glob('*.h5ad'))
print(f'Found {len(h5ad_files)} h5ad files:\n')

# Group by sample prefix
from collections import defaultdict
groups = defaultdict(dict)

for f in h5ad_files:
    name = f.stem
    # Split into sample + modality
    if name.startswith('E13_20_fig3_'):
        sample = 'E13_20_fig3'
        modality = name.replace('E13_20_fig3_', '')
    elif name.startswith('E13_50_2_'):
        sample = 'E13_50_2'
        modality = name.replace('E13_50_2_', '')
    elif name.startswith('E13_50_1_'):
        sample = 'E13_50_1'
        modality = name.replace('E13_50_1_', '')
    else:
        print(f'  Skipping: {name}')
        continue
    groups[sample][modality] = f
    print(f'  {sample} / {modality}: {f.name}')

print()
for sample, mods in groups.items():
    print(f'{sample}: {list(mods.keys())}')

## 2. Explore each sample group

In [ ]:
for sample, mods in groups.items():
    print(f'\n{"=" * 60}')
    print(f'Sample: {sample}')
    print(f'{"=" * 60}')
    for modality, path in mods.items():
        a = sc.read_h5ad(path)
        sp_keys = list(a.obsm.keys())
        is_sparse = sparse.issparse(a.X)
        xmax = a.X.max() if is_sparse else np.max(a.X)
        xmin = a.X.min() if is_sparse else np.min(a.X)
        print(f'\n  {modality}:')
        print(f'    Shape: {a.shape}')
        print(f'    Sparse: {is_sparse}')
        print(f'    X range: [{xmin:.2f}, {xmax:.2f}]')
        print(f'    obs cols: {list(a.obs.columns)}')
        print(f'    obsm keys: {sp_keys}')
        print(f'    var names unique: {a.var_names.is_unique}')
        print(f'    var[:5]: {list(a.var_names[:5])}')
        # Show spatial coord range
        for sk in sp_keys:
            coords = a.obsm[sk]
            print(f'    {sk} range: x=[{coords[:,0].min():.0f}, {coords[:,0].max():.0f}], '
                  f'y=[{coords[:,1].min():.0f}, {coords[:,1].max():.0f}]')
        del a

## 3. Check barcode and gene overlap within groups

In [ ]:
for sample, mods in groups.items():
    print(f'\n--- {sample} ---')
    obs_sets = {}
    var_sets = {}
    for modality, path in mods.items():
        a = sc.read_h5ad(path)
        a.var_names_make_unique()
        obs_sets[modality] = set(a.obs_names)
        var_sets[modality] = set(a.var_names)
        del a

    # Barcode overlap
    mod_names = list(obs_sets.keys())
    common_obs = obs_sets[mod_names[0]]
    for m in mod_names[1:]:
        common_obs = common_obs & obs_sets[m]
    print(f'  Barcodes per modality: {{", ".join(f"{m}: {len(obs_sets[m])}" for m in mod_names)}}')
    print(f'  Common barcodes: {len(common_obs)}')

    # Gene overlap (only meaningful between epigenetic modalities or RNA vs epigenetic)
    for i, m1 in enumerate(mod_names):
        for m2 in mod_names[i+1:]:
            overlap = var_sets[m1] & var_sets[m2]
            print(f'  Gene overlap {m1}({len(var_sets[m1])}) ∩ {m2}({len(var_sets[m2])}): {len(overlap)}')

## 4. Visualize spatial layout per sample

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (sample, mods) in zip(axes, groups.items()):
    # Use first modality to show spatial layout
    first_mod = list(mods.keys())[0]
    a = sc.read_h5ad(mods[first_mod])
    # Pick spatial key
    if 'spatial' in a.obsm:
        coords = a.obsm['spatial']
    elif 'spatial_pxl' in a.obsm:
        coords = a.obsm['spatial_pxl']
    else:
        coords = list(a.obsm.values())[0]

    ax.scatter(coords[:, 0], coords[:, 1], s=1, alpha=0.5)
    ax.set_title(f'{sample}\n({a.n_obs} spots, {first_mod})')
    ax.set_aspect('equal')
    ax.invert_yaxis()
    del a

plt.tight_layout()
plt.show()

## 5. Process RNA per sample

For samples with RNA: normalize, log-transform, find HVGs, PCA, neighbors, UMAP, leiden clustering.
This gives us a properly analyzed RNA object before adding epigenetic features.

In [ ]:
def get_spatial(adata):
    """Extract spatial coords from whichever obsm key exists."""
    if 'spatial' in adata.obsm:
        return adata.obsm['spatial']
    elif 'spatial_pxl' in adata.obsm:
        return adata.obsm['spatial_pxl']
    return list(adata.obsm.values())[0]


def process_rna(sample_id, rna_path):
    """Load RNA, normalize, cluster, UMAP."""
    adata = sc.read_h5ad(rna_path)
    adata.var_names_make_unique()

    # Set spatial
    spatial = get_spatial(adata)
    adata.obsm.clear()
    adata.obsm['spatial'] = np.array(spatial, dtype=np.float64)

    # Store raw counts
    adata.layers['counts'] = adata.X.copy()

    # Normalize and log-transform
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

    # HVG, PCA, neighbors, UMAP, leiden
    sc.pp.highly_variable_genes(adata, n_top_genes=2000)
    sc.tl.pca(adata, use_highly_variable=True)
    sc.pp.neighbors(adata, n_pcs=30)
    sc.tl.umap(adata)
    sc.tl.leiden(adata, resolution=0.5)

    adata.obs['sample_id'] = sample_id
    adata.var['modality'] = 'RNA'
    adata.var['gene'] = adata.var_names.tolist()

    print(f'  {sample_id} RNA: {adata.shape}, {adata.obs["leiden"].nunique()} clusters')
    return adata

In [ ]:
rna_processed = {}
for sample, mods in groups.items():
    if 'RNA' in mods:
        print(f'\n--- Processing {sample} RNA ---')
        rna_processed[sample] = process_rna(sample, mods['RNA'])
    else:
        print(f'\n--- {sample}: no RNA, skipping ---')

## 6. RNA QC: UMAP and spatial clustering

In [ ]:
for sample, adata in rna_processed.items():
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # UMAP colored by leiden
    sc.pl.umap(adata, color='leiden', ax=axes[0], show=False, title=f'{sample} UMAP (leiden)')

    # Spatial colored by leiden
    coords = adata.obsm['spatial']
    cats = adata.obs['leiden'].astype('category')
    for cat in cats.cat.categories:
        mask = cats == cat
        axes[1].scatter(coords[mask, 0], coords[mask, 1], s=2, alpha=0.7, label=cat)
    axes[1].set_title(f'{sample} spatial (leiden)')
    axes[1].set_aspect('equal')
    axes[1].invert_yaxis()
    axes[1].legend(markerscale=3, fontsize=7, loc='center left', bbox_to_anchor=(1, 0.5))

    plt.tight_layout()
    plt.show()
    print(f'{sample}: {adata.n_obs} spots, {adata.obs["leiden"].nunique()} clusters')

## 7. Spatial plots of combined data

In [ ]:
for sample, adata in combined.items():
    mods = adata.uns['modalities']
    n_mods = len(mods)
    fig, axes = plt.subplots(1, n_mods, figsize=(5 * n_mods, 5))
    if n_mods == 1:
        axes = [axes]

    coords = adata.obsm['spatial']

    for ax, mod in zip(axes, mods):
        mask = adata.var['modality'] == mod
        X_mod = adata[:, mask].X
        if sparse.issparse(X_mod):
            total_signal = np.array(X_mod.sum(axis=1)).flatten()
        else:
            total_signal = X_mod.sum(axis=1)
        sc_plot = ax.scatter(coords[:, 0], coords[:, 1], c=total_signal,
                            s=2, alpha=0.7, cmap='viridis')
        ax.set_title(f'{sample}\n{mod} total signal')
        ax.set_aspect('equal')
        ax.invert_yaxis()
        plt.colorbar(sc_plot, ax=ax, shrink=0.6)

    plt.tight_layout()
    plt.show()

## 8. Write combined h5ad files

In [ ]:
for sample, adata in combined.items():
    out_path = OUT_DIR / f'{sample}.h5ad'
    adata.write_h5ad(out_path)
    print(f'Wrote {out_path} ({out_path.stat().st_size / 1e6:.1f} MB)')

## 10. Summary table

## 9. Example: single gene across modalities

Show the same gene's signal from each modality side-by-side — this is what KaroSpace will display.

In [ ]:
# Pick a gene present in both RNA and epigenetic modalities
test_gene = 'Sox17'

for sample, adata in combined.items():
    mods = adata.uns['modalities']
    # Build list of feature names to plot for this gene
    feat_names = []
    for mod in mods:
        if mod == 'RNA':
            name = test_gene
        else:
            name = f'{mod}:{test_gene}'
        if name in adata.var_names:
            feat_names.append(name)

    if not feat_names:
        print(f'{sample}: {test_gene} not found in any modality')
        continue

    n = len(feat_names)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
    if n == 1:
        axes = [axes]
    coords = adata.obsm['spatial']

    for ax, feat in zip(axes, feat_names):
        idx = list(adata.var_names).index(feat)
        vals = adata[:, feat].X
        if sparse.issparse(vals):
            vals = np.array(vals.todense()).flatten()
        else:
            vals = np.array(vals).flatten()
        sc_plot = ax.scatter(coords[:, 0], coords[:, 1], c=vals,
                            s=3, alpha=0.8, cmap='magma')
        ax.set_title(f'{sample}\n{feat}')
        ax.set_aspect('equal')
        ax.invert_yaxis()
        plt.colorbar(sc_plot, ax=ax, shrink=0.6)

    plt.suptitle(f'{test_gene} across modalities', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
summary_rows = []
for sample, adata in combined.items():
    for mod in adata.uns['modalities']:
        n_feats = (adata.var['modality'] == mod).sum()
        summary_rows.append({
            'sample': sample,
            'modality': mod,
            'n_cells': adata.n_obs,
            'n_features': n_feats,
        })

pd.DataFrame(summary_rows)